## Notebook36

### Setup

Run all of the following before starting the notebook.

In [ ]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds2/refs/heads/main/funs.py

In [ ]:
! pip install requests-cache --quiet

In [ ]:
import time

import polars as pl
from plotnine import ggplot, aes
from polars import col as c

import funs

import requests
import requests_cache

In [ ]:
session = requests_cache.CachedSession(
    cache_name="data/turing/requests_cache",
    backend="sqlite",
    allowable_methods=("GET", "HEAD", "POST"),
    allowable_codes=(200, 404),
    expire_after=None
)

base_url = "https://en.wikipedia.org/w/api.php"
headers = {
    "User-Agent": "DataScienceBook/1.0 (yourname@example.edu)"
}

### Questions

This notebook picks up the Turing Award laureate corpus notebook35 started:
the same 81 recipients of `Category:Turing Award laureates`, the same
`data/turing/requests_cache` cache file, and the same identifier pair
(`page_id` stable, `doc_id` readable). Nothing here starts a second
population; every question below attaches new data to the 81 rows already
defined. That includes rebuilding `laureates` itself in question 1 exactly as
notebook35 did, since a notebook cannot assume another notebook's Python
session is still around, only its cache file. Print results unless a
question says otherwise.

1. In the first block, rewrite `polite_get` exactly as in notebook35. In the
second block, send one `list="categorymembers"` request for `"Category:Turing
Award laureates"` on its own and print `.from_cache` for that single call. In
the third block, rewrite `category_members` exactly as in notebook35, rebuild
the full `laureates` table, and print its shape.

In [ ]:
def polite_get(url, params=None):
    session.cookies.clear()
    response = session.get(url, params=params, headers=headers)
    response.raise_for_status()
    if not getattr(response, "from_cache", False):
        time.sleep(0.2)
    return response

In [ ]:
params = {
    "action": "query",
    "format": "json",
    "list": "categorymembers",
    "cmtitle": "Category:Turing Award laureates",
    "cmtype": "page",
    "cmlimit": "max"
}
first_call = polite_get(base_url, params)
first_call.from_cache

In [ ]:
def category_members(category):
    members = []
    params = {
        "action": "query",
        "format": "json",
        "list": "categorymembers",
        "cmtitle": category,
        "cmtype": "page",
        "cmlimit": "max"
    }
    while True:
        data = polite_get(base_url, params).json()
        members.extend(data["query"]["categorymembers"])
        if "continue" not in data:
            break
        params.update(data["continue"])
    return members

members = category_members("Category:Turing Award laureates")

laureates = (
    pl.DataFrame([
        {"page_id": m["pageid"], "doc_id": m["title"]}
        for m in members
    ])
    .unique(subset="page_id", keep="first")
    .sort(c.doc_id)
)
laureates.select(pl.len())

**Answer**:


2. In the first block, write a small `add_doc_id` helper (a join back to
`laureates`, keyed on `page_id`, with both key columns moved to the front). In
the second block, collect each laureate's outgoing internal links with
`prop="links"`, `plnamespace=0`, `pllimit="max"`, paginating with `continue`
the same way category members did, attach `doc_id` with `add_doc_id`, and print
the shape of the resulting `links` table.

In [ ]:
def add_doc_id(df, spine=None):
    spine = laureates if spine is None else spine
    return (
        df
        .join(spine.select(c.page_id, c.doc_id), on=c.page_id)
        .select(c.page_id, c.doc_id, pl.exclude("page_id", "doc_id"))
    )

In [ ]:
rows = []
for page_id in laureates["page_id"]:
    params = {
        "action": "query",
        "format": "json",
        "prop": "links",
        "plnamespace": 0,
        "pllimit": "max",
        "pageids": page_id
    }
    while True:
        data = polite_get(base_url, params).json()
        page = data["query"]["pages"][str(page_id)]
        for link in page.get("links", []):
            rows.append({"page_id": page_id, "link_title": link["title"]})
        if "continue" not in data:
            break
        params.update(data["continue"])

links = add_doc_id(pl.DataFrame(rows))
links.select(pl.len())

**Answer**:


3. In the first block, join `links` back onto `laureates` (matching
`link_title` to `doc_id`) to keep only the edges that land on another laureate,
and print its row count. In the second block, group the result by `doc_id`,
count edges per laureate into `n_links`, and print `.describe()` for that
column. Something about the numbers should look wrong before you even check a
single row. In the third block, use `prop="templates"` on one flagged
laureate's page to find the explanation, and print the offending template's
title.

In [ ]:
links_internal = links.join(
    laureates.select(
        link_title = c.doc_id,
        to_page_id = c.page_id,
        to_doc_id = c.doc_id
    ),
    on=c.link_title
)
links_internal.select(pl.len())

In [ ]:
(
    links_internal
    .group_by(c.doc_id)
    .agg(pl.len().alias("n_links"))
    .select(c.n_links)
    .describe()
)

In [ ]:
params = {
    "action": "query",
    "format": "json",
    "prop": "templates",
    "titles": "Martin Hellman",
    "tllimit": "max"
}
data = polite_get(base_url, params).json()
page = list(data["query"]["pages"].values())[0]
[t["title"] for t in page["templates"] if "Turing" in t["title"]]

**Answer**:


4. In the first block, collect the full revision history for every laureate
(`prop="revisions"`, `rvprop="ids|timestamp|user|size|comment"`,
`rvlimit="max"`, paginating on `continue`), parse `timestamp` into a real
datetime, and print the shape. In the second block, group by `doc_id`, take the
earliest `timestamp` per laureate as that article's creation date, and print
the 5 earliest-created articles. In the third block, print the 5 most recently
created.

In [ ]:
rows = []
for page_id in laureates["page_id"]:
    params = {
        "action": "query",
        "format": "json",
        "prop": "revisions",
        "rvprop": "ids|timestamp|user|size|comment",
        "rvlimit": "max",
        "pageids": page_id
    }
    while True:
        data = polite_get(base_url, params).json()
        page = data["query"]["pages"][str(page_id)]
        for rev in page.get("revisions", []):
            rows.append({
                "page_id": page_id,
                "rev_id": rev["revid"],
                "timestamp": rev["timestamp"],
                "user": rev.get("user"),
                "size": rev["size"],
                "comment": rev.get("comment")
            })
        if "continue" not in data:
            break
        params.update(data["continue"])

revisions = (
    pl.DataFrame(rows)
    .pipe(add_doc_id)
    .with_columns(
        timestamp = c.timestamp.str.to_datetime("%Y-%m-%dT%H:%M:%SZ")
    )
)
revisions.select(pl.len())

In [ ]:
created = (
    revisions
    .group_by(c.doc_id)
    .agg(c.timestamp.min().alias("created"))
    .sort(c.created)
)
created.head(5)

In [ ]:
created.tail(5)

**Answer**:


5. In the first block, collect daily page views for every laureate from the
REST API over the same 2025 window the chapter uses (`all-access`, `user`,
`daily`, `20250101` to `20251231`), skipping any 404 the way the chapter does,
and print the shape. In the second block, sum `views` per laureate and print
the 5 highest totals.

In [ ]:
from urllib.parse import quote

pv_base = (
    "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
    "en.wikipedia/all-access/user/{article}/daily/20250101/20251231"
)

rows = []
for doc_id in laureates["doc_id"]:
    article = quote(doc_id.replace(" ", "_"), safe="")
    session.cookies.clear()
    response = session.get(pv_base.format(article=article), headers=headers)
    if not getattr(response, "from_cache", False):
        time.sleep(0.2)
    if response.status_code == 404:
        continue
    response.raise_for_status()
    for item in response.json()["items"]:
        rows.append({
            "article": item["article"],
            "date": item["timestamp"],
            "views": item["views"]
        })

views = (
    pl.DataFrame(rows)
    .with_columns(date = c.date.str.slice(0, 8).str.to_date("%Y%m%d"))
    .join(
        laureates.select(
            c.page_id,
            article = c.doc_id.str.replace_all(" ", "_")
        ),
        on=c.article
    )
    .select(c.page_id, c.date, c.views)
    .pipe(add_doc_id)
)
views.select(pl.len())

In [ ]:
(
    views
    .group_by(c.doc_id)
    .agg(c.views.sum().alias("total_views"))
    .sort(c.total_views, descending=True)
    .head(5)
)

**Answer**:


6. Plot Geoffrey Hinton's daily 2025 page views with `geom_line()`, save it,
and look at it. Hinton's total comfortably beat Tim Berners-Lee's in
question 5 despite Berners-Lee's article having a 15-year head start and
far more interwiki reach (144 language editions, from question 7, against
65 for Hinton). What does the rest of the corpus already on hand suggest
about why, and what does the plot itself leave unexplained?

In [ ]:
hinton = views.filter(c.doc_id == "Geoffrey Hinton")

(
    ggplot(hinton, aes(c.date, c.views))
    .geom_line()
)

**Answer**:


7. In the first block, collect `langlinks` for every laureate (batched 50 page
IDs per request, paginating with `continue`), count distinct languages per
laureate, and print the 6 laureates with the most language editions. In the
second block, print the 6 with the fewest.

In [ ]:
ids = laureates["page_id"].to_list()
batches = [ids[i:(i + 50)] for i in range(0, len(ids), 50)]

rows = []
for batch in batches:
    params = {
        "action": "query",
        "format": "json",
        "prop": "langlinks",
        "lllimit": "max",
        "pageids": "|".join(str(i) for i in batch)
    }
    while True:
        data = polite_get(base_url, params).json()
        for page in data["query"]["pages"].values():
            for link in page.get("langlinks", []):
                rows.append({
                    "page_id": page["pageid"],
                    "lang": link["lang"],
                    "lang_title": link["*"]
                })
        if "continue" not in data:
            break
        params.update(data["continue"])

langlinks = add_doc_id(pl.DataFrame(rows))

lang_counts = (
    langlinks
    .group_by(c.doc_id)
    .agg(pl.len().alias("n_langs"))
    .sort(c.n_langs, descending=True)
)
lang_counts.head(6)

In [ ]:
lang_counts.sort(c.n_langs).head(6)

**Answer**:


8. A hand-typed list of names, unlike a category listing, comes with typos,
casual capitalization, and titles that belong to someone else entirely.
Write `resolve_titles(titles)`: send one batched request with `redirects=1`,
walk the `normalized` and `redirects` maps until they stop changing (exactly
as the book does for its second corpus), and return each title's resolved
title and page ID. Run it on `"donald knuth"`, `"Edsger_W_Dijkstra"`,
`"Herbert Simon"`, `"Alan Turing"`, and `"Not A Real Laureate Page"`, then
add a column checking whether each resolved `page_id` is one of `laureates`'
own. What happened to each of the five, and why isn't "resolves to a real,
unambiguous page" the same test as "belongs to this population"?

In [ ]:
def resolve_titles(titles):
    rows = []
    params = {
        "action": "query",
        "format": "json",
        "prop": "info",
        "redirects": 1,
        "titles": "|".join(titles)
    }
    data = polite_get(base_url, params).json()

    followed = {}
    for kind in ("normalized", "redirects"):
        for step in data["query"].get(kind, []):
            followed[step["from"]] = step["to"]

    pages = {p["title"]: p for p in data["query"]["pages"].values()}
    for title in titles:
        final = title
        while final in followed:
            final = followed[final]
        rows.append({
            "doc_id": title,
            "resolved": final,
            "page_id": pages.get(final, {}).get("pageid")
        })
    return pl.DataFrame(rows)

titles = [
    "donald knuth",
    "Edsger_W_Dijkstra",
    "Herbert Simon",
    "Alan Turing",
    "Not A Real Laureate Page"
]
resolved = resolve_titles(titles)

resolved.with_columns(
    in_laureates = c.page_id.is_in(laureates["page_id"].to_list())
)

**Answer**:


9. In the first block, rebuild `texts` exactly as notebook35 did (it should be
fast; the cache already has every request) and print its shape. In the second
block, write `laureates`, `texts`, `links`, `revisions`, `views`, and
`langlinks` to `data/turing/` as Parquet files. In the third block, write a
short manifest recording the retrieval date, the population definition, and
each table's row count, and print the manifest.

In [ ]:
rows = []
for page_id in laureates["page_id"]:
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": 1,
        "pageids": page_id
    }
    data = polite_get(base_url, params).json()
    page = data["query"]["pages"][str(page_id)]
    rows.append({"page_id": page_id, "text": page.get("extract", "")})

texts = (
    pl.DataFrame(rows)
    .join(laureates, on=c.page_id)
    .select(c.page_id, c.doc_id, c.text)
)
texts.select(pl.len())

In [ ]:
from pathlib import Path

Path("data/turing").mkdir(parents=True, exist_ok=True)

laureates.write_parquet("data/turing/laureates.parquet")
texts.write_parquet("data/turing/texts.parquet")
links.write_parquet("data/turing/links.parquet")
revisions.write_parquet("data/turing/revisions.parquet")
views.write_parquet("data/turing/views.parquet")
langlinks.write_parquet("data/turing/langlinks.parquet")

In [ ]:
from datetime import date

manifest = f"""# Turing Award laureate corpus

Retrieved: {date.today().isoformat()}
Source: MediaWiki Action API (en.wikipedia.org) and
  Wikimedia REST API v1 (wikimedia.org/api/rest_v1)
Population: direct article members of Category:Turing Award laureates
  (no subcategories)
Pages: {len(laureates)}
Revisions: {len(revisions)}
Pageview window: 2025-01-01 to 2025-12-31 (all-access, user agent class)
Tables: laureates, texts, links, revisions, views, langlinks
Raw responses: data/turing/requests_cache.sqlite
"""

with open("data/turing/MANIFEST.md", "w") as f:
    f.write(manifest)

print(manifest)

**Answer**:


10. In the first block, pull the most recent `rev_id` for `"Donald Knuth"` out
of `revisions`, then request that exact revision's wikitext with `revids=` and
`prop="revisions"`, `rvprop="ids|timestamp|content"`, `rvslots="main"`, print
the returned timestamp and the first 200 characters, and confirm the timestamp
matches the row `revisions` already had for that ID. In the second block,
resend question 1's population request with `formatversion=2` added and print
the type of `data["query"]["categorymembers"]` next to what it was without the
parameter.
Why does pinning a revision ID matter more here than pinning a format version,
even though the callout on API versioning treats them side by side?

In [ ]:
latest = (
    revisions
    .filter(c.doc_id == "Donald Knuth")
    .sort(c.timestamp, descending=True)
    .head(1)
)
rev_id = latest["rev_id"].item()

params = {
    "action": "query",
    "format": "json",
    "revids": rev_id,
    "rvprop": "ids|timestamp|content",
    "rvslots": "main",
    "prop": "revisions"
}
data = polite_get(base_url, params).json()
page = list(data["query"]["pages"].values())[0]
pinned = page["revisions"][0]
pinned["timestamp"], pinned["slots"]["main"]["*"][:200]

In [ ]:
params_v1 = {
    "action": "query",
    "format": "json",
    "list": "categorymembers",
    "cmtitle": "Category:Turing Award laureates",
    "cmtype": "page",
    "cmlimit": "max"
}
params_v2 = params_v1 | {"formatversion": 2}

data_v1 = polite_get(base_url, params_v1).json()
data_v2 = polite_get(base_url, params_v2).json()

type(data_v1["query"]["categorymembers"]), type(data_v2["query"]["categorymembers"])

**Answer**:


11. Between notebook35 and this one, the corpus grew from a page ID and a
title to six tables covering text, links, revisions, views, and
languages, all traceable back to one category listing. Name one place in
this corpus where that population definition (direct members of a single,
official award category, no subcategories) worked in the corpus's favor
compared to the chapter's own novelist categories, and one place a
different population would have changed an answer you printed in this
notebook.

**Answer**:
